# finance / portfolio / utility

Constant relative risk aversion utility of wealth, with the risk aversion fitted.

| Term | In This Notebook |
|---|---|
| **Intent** | Turn a wealth multiple into utility so a portfolio can be ranked by what it is worth to you and not by its average. Placeholder: the risk aversion is recovered from synthetic utilities |
| **Project Flavour** | `pyfunc`, local to the notebook (`model.md` 3.2) |
| **Data Splits** | `training` ten wealth multiples with their utility; `validation` five larger; `backtest` ten across the range, held back for acceptance; `golden` the exact wealth multiples that become `golden.json` |

## Setup

- **Stage:** none, preparation for the stages below.
- **Intent:** put `workbench` on the path, load `env`, and declare the model name, dataset names and signature names as constants (`model.md` 6.6).

In [ ]:
import json
from os.path import join

from workbench.config import NOTEBOOK_DIR
from workbench.data import ensure_data, golden_data
from workbench.env import init_env
from workbench.project import provenance, record_project, scratch_project_dir

env = init_env()

WORKSPACE = "finance/portfolio/utility"
MODEL_NAME = "finance_portfolio_utility"
TARGETS = [{"kind": "local"}]
TRAINING = "portfolio_utility_training_2026-11-05_2026-11-05"
VALIDATION = "portfolio_utility_validation_2026-11-05_2026-11-05"
BACKTEST = "portfolio_utility_backtest_2026-11-05_2026-11-05"
GOLDEN = "portfolio_utility_golden"
INPUTS = ["finance_portfolio_utility_wealth"]
OUTPUTS = ["finance_portfolio_utility_value"]

## Capture Data

- **Stage:** 1, capture data (`model.md` 7.1).
- **Intent:** write the `training`, `validation` and `backtest` files only when they are absent (`model.md` 6.5). The source is synthetic, utility of wealth at a risk aversion of 3, so no query runs.

In [ ]:
from math import log


def utility(wealth, gamma=3.0):
    return log(wealth) if gamma == 1.0 else wealth ** (1.0 - gamma) / (1.0 - gamma)


def labelled(wealths):
    return {INPUTS[0]: wealths, OUTPUTS[0]: [round(utility(wealth), 6) for wealth in wealths]}


training_pl, training_path = ensure_data(WORKSPACE, "training", TRAINING, lambda: labelled([0.5, 0.75, 1.0, 1.25, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0]))
validation_pl, validation_path = ensure_data(WORKSPACE, "validation", VALIDATION, lambda: labelled([6.0, 7.0, 8.0, 9.0, 10.0]))
backtest_pl, backtest_path = ensure_data(WORKSPACE, "backtest", BACKTEST, lambda: labelled([0.6, 0.9, 1.1, 1.7, 2.2, 3.3, 3.7, 4.5, 4.8, 12.0]))
print("training", training_pl.height, "validation", validation_pl.height, "backtest", backtest_pl.height)

## Fit

- **Stage:** 3, fit.
- **Intent:** fit the risk aversion by least squares on `training` only (R6). It is the single number that shapes every later decision, so the plan treats eliciting it as the real work.

In [ ]:
from scipy.optimize import curve_fit

fitted_gamma = curve_fit(lambda wealth, gamma: wealth ** (1.0 - gamma) / (1.0 - gamma), training_pl[INPUTS[0]].to_numpy(), training_pl[OUTPUTS[0]].to_numpy(), p0=[2.0], bounds=(1.5, 10.0))[0][0]
PARAMS = {"gamma": round(float(fitted_gamma), 2)}
print("fitted", PARAMS)
assert PARAMS == {"gamma": 3.0}, "the fit did not recover the risk aversion"

## Evaluate

- **Stage:** 4, evaluate.
- **Intent:** write `model.py` from the fitted values, build the signature, run it through the library runtime and check the `validation` and `backtest` error against the limit the notebook owns (R4, R14).

In [ ]:
from workbench.model import build_signature, evaluate_acceptance, measure, signature_input, signature_output
from workbench.replay import model_predict

MODEL_TEMPLATE = """from math import log

import pandas as pd
from mlflow.models import set_model
from mlflow.pyfunc import PythonModel

INPUT = "finance_portfolio_utility_wealth"
OUTPUT = "finance_portfolio_utility_value"
GAMMA = {gamma!r}


def utility(wealth):
    return log(wealth) if GAMMA == 1.0 else wealth ** (1.0 - GAMMA) / (1.0 - GAMMA)


class Utility(PythonModel):
    def predict(self, context, model_input: pd.DataFrame, params=None):
        return pd.DataFrame({{OUTPUT: [utility(wealth) for wealth in model_input[INPUT]]}})


set_model(Utility())
"""

model_path = join(NOTEBOOK_DIR, WORKSPACE, "model.py")
with open(model_path, "w") as model_file:
    model_file.write(MODEL_TEMPLATE.format(**PARAMS))

signature = build_signature(
    MODEL_NAME,
    [
        signature_input("finance_portfolio_utility_wealth", "none", "Wealth as a multiple of today's wealth."),
    ],
    [
        signature_output("finance_portfolio_utility_value", "none", "Constant relative risk aversion utility of the wealth multiple.", datatype="FP64"),
    ],
)
predict = model_predict(model_path, "pyfunc", signature)

In [ ]:
LIMIT = 0.001
SPLITS = (("validation", validation_pl), ("backtest", backtest_pl))


def acceptance_checks():
    answers = {split: predict({INPUTS[0]: frame[INPUTS[0]].to_list()}) for split, frame in SPLITS}
    return [
        {"split": split, "output": output, "name": "rmse", "value": measure(frame[output].to_list(), answers[split][output])["rmse"], "limit": LIMIT, "direction": "max"}
        for split, frame in SPLITS
        for output in OUTPUTS
    ]


evaluation = evaluate_acceptance(acceptance_checks, {"validation": [validation_path], "backtest": [backtest_path]})
for entry in evaluation["checks"]:
    print(entry["split"], entry["output"], entry["name"], "measured", entry["value"], "passed", entry["passed"])
print("acceptance", "passed" if evaluation["passed"] else "failed, the shipped model is outside the limit")

## Package

- **Stage:** 5, package.
- **Intent:** write the golden file from the shipped runtime, copy `model.py` and the evaluation into a project, verify it and replay the golden through the library runtime (R1, R3, R14).

In [ ]:
def golden_samples():
    xs = [0.5, 1.0, 1.5, 2.0, 5.0, 10.0]
    answers = predict({INPUTS[0]: xs})
    return {INPUTS[0]: [*xs, None], **{output: [*answers[output], None] for output in OUTPUTS}, "expect": ["ok"] * len(xs) + ["error"]}

In [ ]:
from workbench.model import package_project
from workbench.replay import require_replay

golden_pl, golden_path = golden_data(WORKSPACE, GOLDEN, golden_samples)

project_dir = scratch_project_dir(WORKSPACE)
manifest = package_project(
    project_dir,
    MODEL_NAME,
    "pyfunc",
    signature,
    golden_pl.iter_rows(named=True),
    provenance(
        "manual",
        "synthetic constant relative risk aversion utilities, training split",
        [training_path, golden_path],
        "least squares risk aversion",
        implementation="mlflow.pyfunc model.py, requires mlflow and pandas",
    ),
    model_path=model_path,
    evaluation=evaluation,
    sequence="across losses, today's wealth and large gains",
)
print(json.dumps(manifest, indent=2))

require_replay(project_dir)
metadata = record_project(WORKSPACE, project_dir, TARGETS)
print("replayed the golden sequences exactly through the shipped runtime")